<a href="https://colab.research.google.com/github/Sahilgulati2006/ai-sys-des/blob/main/06-adaptation/01-fine-tune-vs-rag-vs-prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adapting the Model: Fine-Tune vs RAG vs Prompt

**Goal:** Make the call an AI engineer is paid to make: when to *change the model* (fine-tune / LoRA) versus change its *inputs* (prompt, RAG), and be able to defend it. Then, optionally, run a real LoRA fine-tune on a free GPU.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Setup

The concept sections (1–4) run on the free Groq API like every other notebook. The two cells below are all they need. The **optional LoRA appendix** at the end needs a GPU runtime and a different stack; it installs its own dependencies and is clearly fenced, so you can read this notebook end-to-end without ever running it.

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client.
2. **Load your API key.** Free key at [console.groq.com](https://console.groq.com/) (no credit card); in Colab add it via the **key icon** → **Add new secret** named exactly `GROQ_API_KEY`, notebook access on. Locally, set `GROQ_API_KEY` in your environment.

(Full walkthrough: [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [1]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.7 MB/s eta 0:00:00


In [2]:
from aien import setup

# Loads GROQ_API_KEY and returns a ready Groq client. Used only by the concept
# sections below; the optional LoRA appendix runs on a separate GPU stack.
client, MODEL = setup()

Groq client ready. MODEL = openai/gpt-oss-120b


## One axis: how much of the model do you actually change?

The four ways to change what a model does line up on a single spectrum: **how deeply they reach into the model itself.** This axis is the one worth memorizing, because it's *technical* and doesn't move with fashion:

```
  change the INPUTS                          change the WEIGHTS
  (model untouched)                          (model modified)
 ┌───────────────┬───────────────┬───────────────────┬──────────────────┐
 │  Prompting    │     RAG        │   LoRA / QLoRA    │  Full fine-tune  │
 │ instructions  │ retrieved facts│ train ~<1% of the │ retrain every    │
 │ & examples    │ at query time  │ weights (adapter) │ weight           │
 └───────────────┴───────────────┴───────────────────┴──────────────────┘
   ← cheaper, faster, more reversible          more powerful, more costly →
```

**Be precise about the words.** Only the right half actually *adapts the model*; it changes weights. Prompting and RAG change the model's **inputs**; the weights never move. And **LoRA is not a peer of fine-tuning. It's a *kind* of fine-tuning** (you still train weights, just a tiny low-rank slice of them instead of all of them). So the honest taxonomy is: two ways to change *inputs*, and two *depths* of the one thing that changes *the model*.

Read the spectrum left-to-right and the working order falls right out. **Reach for the leftmost lever that solves your problem**, because left is cheaper, faster to iterate, and easier to undo:

1. **Prompting** changes the *instructions*. System prompt, few-shot, output format. Zero training, instant iteration, no infrastructure.
2. **RAG** changes the *inputs*. Retrieve relevant facts at query time and put them in the prompt (all of section 03). Weights never change; you're feeding better context.
3. **LoRA fine-tune** changes *a sliver of the weights*. Train small adapters so new *behavior* is baked in (more below).
4. **Full fine-tune** changes *all* the weights. Rarely the right first move for this role; expensive and easy to do worse than a good prompt.

> **🚩 Common mistake —** jumping straight to the right half. It's the most expensive, the slowest to iterate, and the one people assume is the "real" solution because it sounds like ML. The AI engineer's job is usually to talk a stakeholder *left* on this spectrum, not right.

This notebook is about knowing exactly when reaching into the weights is right, and the next cell explains why that bar has *risen* over time.

## Why "reach left first" is the *modern* answer — the pendulum

"Prompt before fine-tune" isn't an eternal law; it's where a pendulum has swung, and knowing *why* it swung is what lets you argue the call.

- **Early LLM era (~2020–2022): fine-tuning was the default.** Base models were weak instruction-followers with small context windows. If you wanted reliable behavior on your task, you fine-tuned; there wasn't a strong enough prompt-only path. Hosted fine-tuning APIs were where the action was.
- **Now: the field swung *left*.** Base models follow instructions well out of the box, context windows are huge (fit the examples/docs right in the prompt), and RAG matured into the standard way to inject knowledge. For the most common need, *"make the model know or do our thing"*, prompt + RAG now wins on cost, speed, and freshness. Fine-tuning got **less popular for that**, to the point that OpenAI is *winding its hosted fine-tuning down for new users*.
- **The nuance that keeps you honest: fine-tuning didn't die, it *specialized*.** It's no longer the go-to for knowledge, but it's the right tool, and in places a *growing* one, for a few specific jobs:
  - **Style / voice / format at scale**, diffuse behavior that's hard to fully pin down in a prompt.
  - **Distilling a small, cheap, fast model.** Fine-tune a tiny model to match a big prompted one on *one* narrow task, then pay a fraction of the per-call cost and latency. This use case is *rising* precisely because inference cost matters at volume.

> **⭐ Key takeaway —** "prompt/RAG before fine-tune" is true today because the *base models and tooling around them* got good, not because fine-tuning is bad. Say it that way in an interview: name the pendulum, then name the specific conditions (style-at-scale, small-model distillation) where you'd still reach right. That's the difference between reciting a rule and understanding it.

## The decision, in one table

The instinct to internalize: **prompting and RAG change what the model *knows right now*; fine-tuning changes how the model *behaves by default*.** Knowledge → context. Behavior → weights.

| You want to change... | Reach for | Why |
|---|---|---|
| **Facts / knowledge** the model lacks (docs, private data, recent events) | **RAG** (or prompt) | Knowledge changes and must be cited; retrieval keeps it fresh and grounded. Fine-tuning bakes facts in as of training day: stale, uncitable, expensive to update. |
| **Format / structure** (always emit this JSON shape, this schema) | **Prompt** first; fine-tune if it must be bulletproof at scale | Prompting + tool-calling (section 01) covers most of it; fine-tuning removes the last few % of format drift on very high volume. |
| **Style / tone / persona** (sound like our brand, our support voice) | **Fine-tune (LoRA)** | Style is diffuse behavior, hard to fully specify in a prompt; a few hundred examples teach it better than paragraphs of instructions. |
| **A narrow, repetitive task** done cheaper/faster (classify, extract, rewrite at huge volume) | **Fine-tune a small model** | A fine-tuned small model can match a prompted big one on one narrow task, at a fraction of the per-call cost and latency. |
| **New skill / reasoning the base model just can't do** | **Fine-tune**, maybe, but validate hard | Rare for this role, and easy to overestimate; often a better base model or better prompting closes the gap first. |

Two rules that fall out of the table:

- **Fine-tuning is the wrong tool for knowledge.** This is the #1 misconception. "The model doesn't know our product docs → let's fine-tune on them" is almost always a mistake: you get an expensive model that has *memorized* docs it can't cite and can't update without retraining. That's RAG's job.
- **They compose.** Real systems often fine-tune for *style/format* AND use RAG for *knowledge* on the same model. It's not either/or.

## What LoRA / QLoRA actually are

Full fine-tuning updates *every* weight in the model. For a 7B model that's ~7 billion numbers, needing many tens of GB of GPU memory and a serious machine. That's what made fine-tuning inaccessible for years.

**LoRA (Low-Rank Adaptation)** is the trick that changed it. Instead of updating the big weight matrices, you *freeze* them and train tiny "adapter" matrices alongside, typically **<1% of the parameters**. The insight: the *change* you need is low-rank (expressible with far fewer numbers than the full matrix), so a small adapter captures it. You get most of full fine-tuning's quality at a fraction of the memory and storage.

- **LoRA** freezes the base model and trains small low-rank adapters. Adapters are a few MB; you can keep many per base model and swap them.
- **QLoRA** runs LoRA on top of a **quantized** (4-bit) base model. Cuts memory further, enough to fine-tune a 7B model on a single free-tier Colab GPU.
- **PEFT** is HuggingFace's library (Parameter-Efficient Fine-Tuning) that implements LoRA/QLoRA and friends. This is the standard tooling today.

Why this matters for you: LoRA is now *mainstream*, not exotic. "We'll fine-tune a LoRA" is a normal sentence in this job, and adapters are cheap enough to be a real option, which makes knowing *when they're the wrong option* more valuable, not less. (Some inference providers, Groq included, can even **serve** LoRA adapters at inference time, so the trained adapter is deployable, not just a lab artifact.)

## The second axis: full fine-tuning vs LoRA

The spectrum at the top had *two* boxes on the weight-changing side for a reason. Once you've decided to fine-tune, there's a second decision, **how** you update the weights, and it has bigger practical consequences than most people expect.

Keep the words straight: **fine-tuning is the umbrella** (updating a pretrained model on your data); **full fine-tuning (FFT)** and **LoRA** are two *methods* under it. FFT updates every weight; LoRA freezes the base and trains tiny low-rank adapter matrices injected into the layers.

| Aspect | Full fine-tuning (FFT) | LoRA (Low-Rank Adaptation) |
|---|---|---|
| **How it works** | Updates 100% of parameters directly | Freezes base weights; trains small injected adapters |
| **Trainable params** | 100% (e.g. ~8B for Llama-8B) | ~0.1–1% (tens of millions) |
| **VRAM / GPU cost** | Very high: multi-GPU, optimizer states + gradients for every weight | Low, fits a single, even consumer, GPU |
| **Output artifact** | A whole new multi-GB model checkpoint | A small adapter file (often <100 MB) |
| **Catastrophic forgetting** | Higher risk; can erode the base model's general knowledge | Lower risk, since base weights stay frozen |

**Why LoRA is the default in practice.** For LLM work, LoRA (and its quantized cousin QLoRA) is the method teams actually reach for, for two architectural reasons that go beyond "it's cheaper to train":

- **Multi-tenant serving.** Because the base is frozen, you load *one* copy of a big base model into GPU memory and **hot-swap tiny adapters per request** (one for legal summaries, one for code, one for brand voice) instead of standing up a separate full model per task. AWS's [multi-LoRA-on-vLLM writeup](https://aws.amazon.com/blogs/machine-learning/efficiently-serve-dozens-of-fine-tuned-models-with-vllm-on-amazon-sagemaker-ai-and-amazon-bedrock/) shows this concretely: adapters "swapped in and out per request" over a shared frozen base, so (their example) five customers each using ~10% of a GPU collapse onto a *single* shared GPU instead of five idle ones.
- **Accessibility.** FFT of a modern LLM needs a multi-GPU cluster; LoRA lets a team adapt an open-weight model on a fraction of that hardware. That's most of why fine-tuning became approachable at all (see the QLoRA-on-a-free-T4 appendix below).

**When to pick which:**

- **LoRA** suits teaching an already-capable base model a *style, tone, structured format* (strict JSON/SQL), or a domain task. This is the overwhelming majority of fine-tuning you'll actually do.
- **Full fine-tuning** suits drastic, foundational change: adding a new language, shifting deep reasoning behavior, heavily reshaping domain representations where a thin adapter isn't enough. Rare for this role.

> **⭐ Key takeaway —** "should we fine-tune?" and "FFT or LoRA?" are two different questions. In practice the answer to the second is almost always **LoRA**, not just to save training cost, but because a frozen base you can swap adapters against is a fundamentally better *serving* story.

> **⚠️ Production reality —** where the adapter lives depends on the platform, and this bites people. Multi-LoRA serving (above) keeps adapters **separate, loaded on demand**. But some hosted paths want them **merged into the base first**: e.g. [Bedrock Custom Model Import](https://aws.amazon.com/blogs/machine-learning/amazon-bedrock-custom-model-import-now-generally-available/) imports *full merged* Safetensors weights and (verbatim) "expects the adapters to be merged into the main base model weight," not a bare adapter. Same LoRA, two very different deployment shapes; know which one your target platform expects. (This is the same fact behind the Bedrock note in the provider table below.)

### If you *do* fine-tune: hosted API vs train-it-yourself, and who lets you touch LoRA

Once an eval says fine-tuning earns its place, there are two ways to get it done. And there's a subtlety worth knowing before a design conversation: **not every provider lets you actually control LoRA.** Some abstract the training method away entirely; others make the adapter a first-class thing you configure and deploy. All snippets below are **reference only, not runnable here** (they need each vendor's SDK + a paid key); they're shown so the *shapes* are familiar.

**Path 1 — Hosted fine-tuning, method abstracted (OpenAI, AWS Bedrock).** You upload a `.jsonl`, call one method, and the provider trains *and* hosts the result. You do **not** choose LoRA vs full fine-tune; the technique is hidden. You pick the *objective* and some training knobs, nothing more. This is the common production path when you just want a better model, not a research knob.

```python
# OpenAI — reference only. Note: OpenAI is winding hosted fine-tuning DOWN for new users.
# You choose the OBJECTIVE (method=supervised|dpo|reinforcement) — never "LoRA".
from openai import OpenAI
client = OpenAI()

f = client.files.create(file=open("train.jsonl", "rb"), purpose="fine-tune")
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini-2025-04-14",
    training_file=f.id,
    method={"type": "supervised"},          # the only "how" you control; no rank/adapter
)
job = client.fine_tuning.jobs.retrieve(job.id)     # poll until status == "succeeded"
# then call it like any model:
# client.chat.completions.create(model=job.fine_tuned_model, messages=[...])
```

```python
# AWS Bedrock — reference only. Also fully abstracted: no LoRA type, no lora_rank.
# customizationType is FINE_TUNING | CONTINUED_PRE_TRAINING | DISTILLATION | ...  (no "LoRA")
import boto3
bedrock = boto3.client("bedrock")           # control plane

bedrock.create_model_customization_job(
    jobName="my-ft-job", customModelName="my-model",
    roleArn="arn:aws:iam::123456789012:role/BedrockCustomizationRole",
    baseModelIdentifier="meta.llama3-1-8b-instruct-v1:0:128k",
    customizationType="FINE_TUNING",
    hyperParameters={"epochCount": "5", "batchSize": "1", "learningRate": "0.0001"},
    trainingDataConfig={"s3Uri": "s3://my-bucket/train.jsonl"},
    outputDataConfig={"s3Uri": "s3://my-bucket/output/"},
)
# The custom model is then DEPLOYED (custom-model deployment or provisioned throughput)
# and invoked by its ARN. To bring your OWN LoRA to Bedrock you must MERGE it into the
# base weights and import the full model — you can't upload a bare adapter. Explicit,
# rank-controllable LoRA on AWS lives in SageMaker, not Bedrock.
```

**Path 2 — Hosted fine-tuning with *explicit* LoRA (Together, Fireworks).** These inference-first platforms make the LoRA adapter first-class: you choose the **rank** (and often alpha/dropout), they train the adapter, and you deploy it to an endpoint and call it. This is the real "make a LoRA via API and use it" path.

```python
# Together AI — reference only. LoRA is explicit and is the DEFAULT training type.
from together import Together
client = Together()

tf = client.files.upload(file="train.jsonl", purpose="fine-tune")
job = client.fine_tuning.create(
    training_file=tf.id,
    model="Qwen/Qwen3.5-9B",       # check the current supported-models list
    lora=True,                     # <-- you explicitly request LoRA...
    lora_r=8, lora_alpha=16,       # ...and set its rank/alpha (surfaced by the CLI + REST API)
    n_epochs=3,
)
# When done, DEPLOY the adapter to a dedicated endpoint, then call it by the endpoint id:
#   tg beta endpoints deploy <MODEL_OBJECT_ID> --endpoint my-lora
#   Together(base_url="https://api-inference.together.ai/v1").chat.completions.create(
#       model="your-project-slug/my-lora", messages=[...])
```

```bash
# Fireworks AI — reference only (firectl CLI). Supervised fine-tuning IS LoRA here;
# you set the rank directly and address the adapter as a first-class model resource.
firectl dataset create my-data ./train.jsonl
firectl sftj create --base-model <BASE_MODEL_ID> --dataset my-data \
                    --output-model my-lora --lora-rank 16      # <-- explicit LoRA rank
firectl deployment create "accounts/<ACCT>/models/my-lora"     # deploy the adapter
# then call model="accounts/<ACCT>/models/my-lora" via the OpenAI-compatible API
```

**Path 3 — Train the adapter yourself** with `peft`/`trl` on your own GPU: full control (rank, target modules, quantization), runs on open weights, no per-token markup on a hosted custom model, but you own the training *and* the serving. That's the optional appendix at the end of this notebook.

> **⭐ Key takeaway —** "we'll fine-tune" hides a real fork. On **OpenAI / Bedrock** the method is abstracted; you can't pick LoRA. On **Together / Fireworks** (and self-hosted `peft`) LoRA is an explicit, rank-controllable adapter you deploy. Know which you're on before you promise a customer "a LoRA."

Either way the *decision* and the *evaluation* are identical, which is the whole point of this notebook. Whether you rent the training or run it, you fine-tune only when prompt + RAG have demonstrably failed an eval, and you prove it helped with the same harness from section 02 / section 04.

## Cost, and the argument you'll actually have

You will, at some point, be in a room where someone says *"can't we just fine-tune it?"* Here's the honest cost picture to answer with, not to shut it down, but to sequence it right:

| | Prompt / RAG | LoRA fine-tune |
|---|---|---|
| **Upfront cost** | ~none | a labeled dataset (hundreds–thousands of examples) + a GPU training run |
| **Iteration speed** | seconds; edit and rerun | hours; re-curate data, retrain, re-eval |
| **Updating knowledge** | change the retrieved docs | retrain (knowledge is frozen into weights) |
| **Per-call cost at scale** | pay for the context tokens every call | a fine-tuned *small* model can be much cheaper per call |
| **Can you cite sources?** | yes (RAG) | no |
| **Main risk** | prompt bloat, retrieval quality | overfitting, stale knowledge, an eval you didn't build |

The professional move is **prompt → RAG → fine-tune, and only advance when the cheaper lever demonstrably fails an eval.** Fine-tuning without an eval (section 04) is how teams spend a GPU week to make a model *feel* better and *measurably* worse. If you take one thing from this notebook into an interview: you can explain *why you'd usually reach for RAG first, and the specific conditions under which you'd fine-tune anyway* (style/format/latency at volume, rarely knowledge).

In [3]:
# A concept check you CAN run on Groq: does the base model already do the task well
# enough with a good prompt? Answering this honestly is the gate before any fine-tune.
# Here: a formatting/style task, the kind people reach to fine-tune for.
examples = [
    "ticket: wifi keeps dropping in the west conference room since tuesday",
    "ticket: cant log into the vpn, mfa code never arrives",
]
SYSTEM = ('You are a support triage bot. For each ticket reply with ONE line: '
          '"<PRIORITY> | <CATEGORY> | <one-line summary>". '
          'PRIORITY in {P1,P2,P3}. CATEGORY in {network,auth,hardware,other}. Nothing else.')

for t in examples:
    r = client.chat.completions.create(
        model=MODEL, max_tokens=60,
        messages=[{'role': 'system', 'content': SYSTEM},
                  {'role': 'user', 'content': t}],
    )
    print(r.choices[0].message.content.strip())

# If a prompt like this already nails the format across your real ticket distribution,
# you do NOT need to fine-tune for format. Fine-tuning earns its place only when the
# prompt fails often enough, at high enough volume, that baking it in pays for itself.
print('\n-> Prompt got the format. Now ask: does it hold on 500 real tickets? '
      'That eval — not a hunch — is what justifies (or kills) a fine-tune.')




-> Prompt got the format. Now ask: does it hold on 500 real tickets? That eval — not a hunch — is what justifies (or kills) a fine-tune.


---

## Optional appendix: a real LoRA fine-tune (GPU runtime required)

Everything above runs on Groq. **This section does not.** Training weights needs a GPU and a different stack (`peft`, `transformers`, `trl`, `bitsandbytes`), none of which touch Groq. Treat it as a lab you *can* run to make LoRA concrete, not part of the Groq path.

**To run it:** in Colab, **Runtime → Change runtime type → T4 GPU**, then run the cells below. On a CPU runtime the guard cell will stop you with a clear message instead of crashing. This trains a tiny LoRA adapter on a small open model so it finishes in minutes on a free T4; it's a *mechanics* demo, not a quality result.

If you're just here for the judgment (the point of this notebook), you can stop at the line above. You already have it.

In [4]:
# GUARD: this appendix needs a GPU. Stop early and clearly on CPU rather than
# installing a heavy stack and failing deep in a training loop.
import torch
assert torch.cuda.is_available(), (
    'No GPU detected. This appendix requires one: Runtime -> Change runtime type '
    '-> T4 GPU, then re-run. (The concept sections above do not need this.)'
)
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


In [5]:
# Heavy, GPU-only deps. Not part of the Groq stack; installed here on purpose.
# torchao>=0.16 is required by current peft; Colab ships an older one, so pin it up here.
%pip install -q -U transformers peft trl datasets bitsandbytes accelerate "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 33.3 MB/s eta 0:00:00


In [6]:
# A tiny LoRA fine-tune, start to finish, on a small open model.
# Task: teach a fixed reply STYLE (terse, prefixed) from a handful of examples --
# style is exactly what fine-tuning is good at and prompting struggles to pin down.
#
# This cell is written to work on a free T4 and to actually show the style transfer. The
# comments flag choices that matter (each was a real failure mode when getting this to run):
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

BASE = 'HuggingFaceTB/SmolLM2-135M-Instruct'  # tiny: trains in minutes on a T4

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
# fp32, NOT fp16: a 135M model trained in fp16 tends to produce garbage/NaN and degenerate
# greedy output. fp32 fits a T4 easily here and is far more stable at this scale.
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32, device_map='auto')

# Toy dataset: teach a house style -> every answer starts "ACME> " and stays terse.
raw = [
    ("How do I reset my password?", "ACME> Settings > Security > Reset password. Done."),
    ("Is the API down?", "ACME> Check status.acme.io. If red, we're on it."),
    ("How do I export my data?", "ACME> Settings > Data > Export. You'll get a CSV by email."),
    ("Can I get a refund?", "ACME> Yes within 30 days. Reply here with your order id."),
] * 16  # repeat so the tiny model actually picks up the pattern

IM_END = tok.convert_tokens_to_ids("<|im_end|>")

def build(q, a):
    # Prompt built with the SAME template + add_generation_prompt used at inference time, so
    # train and inference formats match byte-for-byte (a mismatch here = the adapter never fires).
    enc = tok.apply_chat_template([{'role': 'user', 'content': q}],
                                  add_generation_prompt=True, tokenize=True, return_dict=True)
    prompt_ids = list(enc['input_ids'])
    # Answer ends with <|im_end|> and no trailing newline, so the model learns to STOP -- otherwise
    # it never emits the stop token and spirals into an endless run of newline tokens.
    answer_ids = tok(a, add_special_tokens=False)['input_ids'] + [IM_END]
    input_ids = prompt_ids + answer_ids
    # Loss ONLY on the answer (prompt labels = -100). Without this, the model spends its capacity
    # re-parroting the system preamble ("...SmolLM...") instead of learning the ACME style.
    labels = [-100] * len(prompt_ids) + answer_ids
    return {'input_ids': input_ids, 'labels': labels, 'attention_mask': [1] * len(input_ids)}

ds = Dataset.from_list([build(q, a) for q, a in raw])
print('one training example (decoded):\n', repr(tok.decode(ds[0]['input_ids'])))

# A bit more LoRA reach (all attention projections) so a 135M model absorbs the style.
peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
                      task_type='CAUSAL_LM')
model = get_peft_model(model, peft_cfg)

def collate(feats):
    maxlen = max(len(f['input_ids']) for f in feats)
    pad = lambda xs, v: [x + [v] * (maxlen - len(x)) for x in xs]
    return {'input_ids': torch.tensor(pad([f['input_ids'] for f in feats], tok.pad_token_id)),
            'labels': torch.tensor(pad([f['labels'] for f in feats], -100)),
            'attention_mask': torch.tensor(pad([f['attention_mask'] for f in feats], 0))}

trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir='lora-out', num_train_epochs=10,
                           per_device_train_batch_size=4, learning_rate=3e-4,
                           logging_steps=10, report_to='none'),
    train_dataset=ds, data_collator=collate)
trainer.train()
model = trainer.model
print('\nTrained a LoRA adapter with',
      sum(p.numel() for p in model.parameters() if p.requires_grad),
      'trainable params (vs', sum(p.numel() for p in model.parameters()), 'total).')

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

one training example (decoded):
 '<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nHow do I reset my password?<|im_end|>\n<|im_start|>assistant\nACME> Settings > Security > Reset password. Done.<|im_end|>'


Step,Training Loss
10,3.919162
20,2.356324
30,0.990670
40,0.251925
50,0.025159
60,0.007914
70,0.003743
80,0.002393
90,0.001949
100,0.001683



Trained a LoRA adapter with 1843200 trainable params (vs 136358208 total).


In [7]:
# Before/after the adapter, same prompt (one the model never saw in training). The adapter
# should push replies toward the "ACME> ..., terse" house style the base model doesn't use.
def reply(prompt):
    enc = tok.apply_chat_template([{'role': 'user', 'content': prompt}],
                                  add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to(model.device)
    n_prompt = enc['input_ids'].shape[1]
    # repetition_penalty + min_new_tokens stop a tiny model from collapsing into a run of
    # newlines before it says anything; eos = <|im_end|> so it halts cleanly at end-of-turn.
    out = model.generate(**enc, max_new_tokens=40, do_sample=False,
                         repetition_penalty=1.3, min_new_tokens=5,
                         eos_token_id=tok.convert_tokens_to_ids("<|im_end|>"),
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][n_prompt:], skip_special_tokens=True).strip()

q = 'How do I change my email address?'
with model.disable_adapter():          # adapter off -> the base model's default voice
    print('BASE :', reply(q))
print('LoRA :', reply(q))              # adapter on -> the learned "ACME> ..." house style

print('\nThis is a mechanics demo on a 135M model and a toy dataset -- do not read quality '
      'into it. The point: <1% of params, a few minutes, a STYLE the base model did not have, '
      'learned from a handful of examples and toggled on/off with one adapter. Whether it BEATS '
      'a good prompt is an eval question (section 04), every time.')

BASE : To update your e-mail account on Facebook or other social media platforms like Instagram and Twitter:

1. Open the Facebook app (if you haven't already).
   - Click "Settings" at
LoRA : ACME> Settings > Email > Edit. You'll get an info sheet with your updates.

This is a mechanics demo on a 135M model and a toy dataset -- do not read quality into it. The point: <1% of params, a few minutes, a STYLE the base model did not have, learned from a handful of examples and toggled on/off with one adapter. Whether it BEATS a good prompt is an eval question (section 04), every time.


## Exercises

1. **Argue it in an interview.** In 3–4 sentences each, answer: (a) *"A customer wants the model to know their internal wiki: fine-tune or RAG?"* (b) *"They want every answer in their brand voice: fine-tune or prompt?"* (c) *"They classify 2M tickets/day and GPT-4-class latency is too slow: what do you propose?"* These are real FDE screening questions.
2. **Kill a fine-tune with a prompt.** Take the house-style task from the appendix and try to get the *base* model (adapter off, or any Groq model) to match the "ACME> terse" style with prompting + few-shot alone. How close can you get? This is the honest first step before any fine-tune.
3. **Cost the decision.** For a narrow classification task at 1M calls/day, estimate monthly cost of (a) prompting a large model vs (b) a fine-tuned small model (reuse the cost math from `01-model-apis/04-context-and-caching`). At what volume does the fine-tune's upfront cost pay back?
4. **FFT or LoRA?** For each, say which method you'd choose and why, in one sentence: (a) match a law firm's house drafting style; (b) teach a model a low-resource language it barely handles; (c) emit strict internal-schema JSON with near-zero drift; (d) shift the model's core reasoning behavior. Which of these are the "rare" full-fine-tune cases?
5. **Design the serving story.** A customer needs *eight* department-specific variants (support, legal, sales, …) of the same base model, each low-traffic. Contrast the cost of eight full fine-tuned models vs one frozen base with eight hot-swapped LoRA adapters (see the multi-LoRA writeup). Why does the frozen-base design win on GPU utilization, and what's the catch if one variant needs a *drastic* behavior change?
6. **(GPU) Vary the LoRA rank.** Re-run the appendix with `r=1` and `r=64`. Does higher rank change the learned style or just the trainable-param count? Relate it to the low-rank insight: how small a `r` still captures this task?